# TRM Experiment Runner

Unified notebook for training, evaluating, and analyzing Tiny Recursive Model (TRM) experiments.

**Tasks supported:** `sudoku` · `maze`  
**Sections:** Environment → Config → Data → Model → Training → Evaluation → Visualization → Ablations → Logging

---

## Section 1: Environment & Setup

In [1]:
import sys
import os

# Notebook lives in experiments/ — add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/hulk/Library/CloudStorage/OneDrive-SNHU/GitHub/Tiny-Recursive-Model-TRM-


In [2]:
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import json
import subprocess
import time
import random
import socket
from datetime import datetime
from pathlib import Path
from collections import defaultdict
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

# Project imports
from model import create_trm_model, SudokuHead, MazeHead, TinyRecursiveModel
from train import TRMTrainer, EMA, TASK_DEFAULTS
from evaluate import evaluate, evaluate_sudoku, evaluate_maze, evaluate_per_step
from data import get_dataset
from data.sudoku import generate_puzzle, difficulty_score as sudoku_difficulty
from data.maze import generate_maze, solve_maze, make_path_target
from viz import print_sudoku, print_maze, plot_per_step_accuracy
from viz.visualize import plot_multi_task_per_step

print('All imports OK')

All imports OK


In [3]:
# Device detection — project targets Apple Silicon (MPS)
try:
    from device import get_device, print_device_info
    print_device_info()
    DEVICE = get_device()
except RuntimeError as e:
    print(f'[WARNING] {e}')
    if torch.cuda.is_available():
        DEVICE = 'cuda'
    else:
        DEVICE = 'cpu'
    print(f'Falling back to: {DEVICE}')

print(f'\nActive device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')
elif DEVICE == 'mps':
    print('MPS (Metal Performance Shaders) — Apple Silicon')

MPS (Metal) GPU READY
PyTorch version : 2.10.0
Device          : mps
MPS available   : True
MPS built       : True

Active device: mps
PyTorch version: 2.10.0
MPS (Metal Performance Shaders) — Apple Silicon


In [4]:
def set_seed(seed: int = 42):
    """Set all RNG seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Uncomment for full determinism (slower):
    # torch.use_deterministic_algorithms(True)
    print(f'Seed set: {seed}')

set_seed(42)

Seed set: 42


## Section 2: Experiment Configuration

All tunable parameters live here. Modify this cell, then run the notebook top-to-bottom.

In [5]:
# ── Task selection ──────────────────────────────────────────────────────────
TASK = 'sudoku'   # 'sudoku' | 'maze'

# Load project defaults, then override below
_d = TASK_DEFAULTS[TASK].copy()

CONFIG = {
    # Identity
    'experiment_name': f'exp_{TASK}_r003',
    'notes': 'baseline run',
    'task': TASK,
    'seed': 42,

    # ── Model architecture ───────────────────────────────────────────────────
    'dim':          _d['dim'],           # embedding / hidden size
    'n_layers':     _d['n_layers'],      # transformer layers (weight-shared)
    'n_heads':      _d['n_heads'],       # attention heads
    'n_recursions': _d['n_recursions'],  # latent recursion depth
    'n_cycles':     _d['n_cycles'],      # deep recursion cycles
    # 'n_cycles':     1,  # TEMP: single cycle for faster runs
    'n_supervision':_d['n_supervision'], # supervision / halting steps
    # 'n_supervision': 4,  # TEMP: fewer supervision steps for faster runs

    # ── Training ─────────────────────────────────────────────────────────────
    'lr':             _d['lr'],
    'weight_decay':   _d['weight_decay'],
    'batch_size':     _d['batch_size'],
    # 'batch_size':     128,  # TEMP: smaller batch size for faster runs
    'num_epochs':     _d['num_epochs'],
    # 'num_epochs':     30,   # TEMP: fewer epochs for faster runs
    'max_steps':      _d['max_steps'],
    # 'max_steps':      5_000,  # TEMP: fewer max steps for faster runs
    'warmup_steps':   _d['warmup_steps'],
    # 'warmup_steps':   200,  # TEMP: fewer warmup steps for faster runs
    'use_ema':        True,
    'ema_decay':      0.999,
    'grad_clip':      1.0,
    'early_stop_patience': 0,   # epochs with no val improvement; 0 = disabled
    'save_every_n_epochs': 10,

    # ── Data ─────────────────────────────────────────────────────────────────
    'num_train':  _d['num_train'],
    # 'num_train':  5_000,  # TEMP: smaller dataset for faster runs
    'num_val':    _d['num_val'],
    # 'num_val':    1_000,  # TEMP: smaller dataset for faster runs
    'augment':    True,
    # Sudoku-specific
    'min_givens': _d.get('min_givens', 17),
    'max_givens': _d.get('max_givens', 35),
    # Maze-specific
    'grid_size':  _d.get('grid_size', 9),

    # ── Paths ─────────────────────────────────────────────────────────────────
    'device': DEVICE,
}

# Derived paths
CONFIG['checkpoint_dir'] = f'experiments/checkpoints/{CONFIG["experiment_name"]}'
Path(CONFIG['checkpoint_dir']).mkdir(parents=True, exist_ok=True)
Path('experiments/results').mkdir(parents=True, exist_ok=True)
Path('experiments/logs').mkdir(parents=True, exist_ok=True)

print('CONFIG ready')

CONFIG ready


In [6]:
print('=' * 55)
print(f'  EXPERIMENT: {CONFIG["experiment_name"]}')
print('=' * 55)
for k, v in sorted(CONFIG.items()):
    print(f'  {k:<28}: {v}')
print('=' * 55)

  EXPERIMENT: exp_sudoku_r003
  augment                     : True
  batch_size                  : 64
  checkpoint_dir              : experiments/checkpoints/exp_sudoku_r003
  device                      : mps
  dim                         : 128
  early_stop_patience         : 0
  ema_decay                   : 0.999
  experiment_name             : exp_sudoku_r003
  grad_clip                   : 1.0
  grid_size                   : 9
  lr                          : 0.0003
  max_givens                  : 35
  max_steps                   : 100000
  min_givens                  : 17
  n_cycles                    : 3
  n_heads                     : 8
  n_layers                    : 2
  n_recursions                : 8
  n_supervision               : 14
  notes                       : baseline run
  num_epochs                  : 100
  num_train                   : 50000
  num_val                     : 5000
  save_every_n_epochs         : 10
  seed                        : 42
  task             

In [7]:
try:
    git_hash = subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'],
        cwd=PROJECT_ROOT, stderr=subprocess.DEVNULL
    ).decode().strip()
    print(f'Git commit : {git_hash}')
    print(f'Short hash : {git_hash[:8]}')
except Exception as e:
    git_hash = 'unknown'
    print(f'Could not get git hash: {e}')

CONFIG['git_hash'] = git_hash

Git commit : e35e5793bd0bed3c8267046fa65984c4c58f4699
Short hash : e35e5793


## Section 3: Data Loading

In [8]:
set_seed(CONFIG['seed'])

# Build kwargs for each task
if CONFIG['task'] == 'sudoku':
    _shared_kwargs = {
        'min_givens': CONFIG['min_givens'],
        'max_givens': CONFIG['max_givens'],
        'augment':    CONFIG['augment'],
    }
else:
    _shared_kwargs = {
        'grid_size': CONFIG['grid_size'],
        'augment':   CONFIG['augment'],
    }

print('Generating datasets (may take a moment)...')
train_ds = get_dataset(CONFIG['task'], 'train', seed=CONFIG['seed'],
                       num_samples=CONFIG['num_train'], **_shared_kwargs)
val_ds   = get_dataset(CONFIG['task'], 'val',   seed=CONFIG['seed'],
                       num_samples=CONFIG['num_val'],   **_shared_kwargs)

print(f'  Train : {len(train_ds):,} samples')
print(f'  Val   : {len(val_ds):,} samples')

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=0)

print(f'  Train batches: {len(train_loader)}')
print(f'  Val batches  : {len(val_loader)}')

Seed set: 42
Generating datasets (may take a moment)...
  Train : 50,000 samples
  Val   : 5,000 samples
  Train batches: 782
  Val batches  : 79


In [9]:
x_s, y_s = train_ds[0]
print(f'Sample shapes — x: {x_s.shape}  y: {y_s.shape}')
print(f'x dtype: {x_s.dtype}  range: [{x_s.min()}, {x_s.max()}]')
print(f'y dtype: {y_s.dtype}  range: [{y_s.min()}, {y_s.max()}]')

if CONFIG['task'] == 'sudoku':
    puzzle_np = x_s.numpy().reshape(9, 9)
    sol_np    = y_s.numpy().reshape(9, 9)
    n_given   = int((puzzle_np > 0).sum())
    n_empty   = 81 - n_given
    print(f'\nGivens: {n_given}  Empty: {n_empty}  Difficulty score: {n_empty}')
    print('\nSample puzzle (solution shown since model not trained yet):')
    print(print_sudoku(puzzle_np, sol_np, sol_np))
else:
    gs = CONFIG['grid_size']
    maze_np = x_s.numpy().reshape(gs, gs)
    path_np = y_s.numpy().reshape(gs, gs)
    print(f'\nMaze ({gs}x{gs})  Path cells: {int(path_np.sum())}  '
          f'Open cells: {int((maze_np == 1).sum())}')
    print('\nSample maze:')
    print(print_maze(maze_np, true_path=path_np))

Sample shapes — x: torch.Size([81])  y: torch.Size([81])
x dtype: torch.int64  range: [0, 9]
y dtype: torch.int64  range: [1, 9]

Givens: 23  Empty: 58  Difficulty score: 58

Sample puzzle (solution shown since model not trained yet):
+-------+-------+-------+
|  7  4  1 |  3  9  5 |  2  6  8 |
|  9  5  6 |  8  2  7 |  1  3  4 |
|  2  8  3 |  6  1  4 |  9  5  7 |
+-------+-------+-------+
|  6  9  4 |  5  3  1 |  8  7  2 |
|  5  1  8 |  7  4  2 |  6  9  3 |
|  3  2  7 |  9  6  8 |  5  4  1 |
+-------+-------+-------+
|  8  3  5 |  2  7  9 |  4  1  6 |
|  1  7  9 |  4  8  6 |  3  2  5 |
|  4  6  2 |  1  5  3 |  7  8  9 |
+-------+-------+-------+
+-------+-------+-------+
|  7  4  1 |  3  9  5 |  2  6  8 |
|  9  5  6 |  8  2  7 |  1  3  4 |
|  2  8  3 |  6  1  4 |  9  5  7 |
+-------+-------+-------+
|  6  9  4 |  5  3  1 |  8  7  2 |
|  5  1  8 |  7  4  2 |  6  9  3 |
|  3  2  7 |  9  6  8 |  5  4  1 |
+-------+-------+-------+
|  8  3  5 |  2  7  9 |  4  1  6 |
|  1  7  9 |  4  8  6 |

## Section 4: Model Instantiation

In [10]:
set_seed(CONFIG['seed'])

model_kwargs = {
    'task':         CONFIG['task'],
    'dim':          CONFIG['dim'],
    'n_layers':     CONFIG['n_layers'],
    'n_heads':      CONFIG['n_heads'],
    'n_recursions': CONFIG['n_recursions'],
    'n_cycles':     CONFIG['n_cycles'],
    'n_supervision':CONFIG['n_supervision'],
}
if CONFIG['task'] == 'maze':
    model_kwargs['grid_size'] = CONFIG['grid_size']

model = create_trm_model(**model_kwargs)
print(f'Model created: task={CONFIG["task"]}')

Seed set: 42
Model created: task=sudoku


In [11]:
total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'{"Total parameters":<30}: {total_p:>12,}')
print(f'{"Trainable parameters":<30}: {trainable_p:>12,}')
print()

_components = {
    'task_head':       model.task_head,
    'layers (shared)': model.layers,
    'norm (output)':   model.output_norm,
    'q_head (halting)':model.q_head,
}
_params = {
    'y_init (scratch)': model.y_init,
    'z_init (scratch)': model.z_init,
}
for name, comp in _components.items():
    n = sum(p.numel() for p in comp.parameters())
    print(f'  {name:<30}: {n:>10,} params')
for name, param in _params.items():
    print(f'  {name:<30}: {param.numel():>10,} params')

Total parameters              :      530,048
Trainable parameters          :      530,048

  task_head                     :      4,736 params
  layers (shared)               :    524,800 params
  norm (output)                 :        128 params
  q_head (halting)              :        128 params
  y_init (scratch)              :        128 params
  z_init (scratch)              :        128 params


In [12]:
model.eval()
with torch.no_grad():
    if CONFIG['task'] == 'sudoku':
        x_dummy = torch.randint(0, 10, (2, 81))
    else:
        gs = CONFIG['grid_size']
        x_dummy = torch.randint(0, 4, (2, gs * gs))

    y_hat              = model(x_dummy, training=False)
    y_hat2, all_steps  = model(x_dummy, training=False, return_all_steps=True)

print(f'Input shape          : {x_dummy.shape}')
print(f'Inference output     : {y_hat.shape}')
print(f'Per-step count       : {len(all_steps)}')
print(f'Per-step step shape  : {all_steps[0].shape}')
print('Forward pass OK')

Input shape          : torch.Size([2, 81])
Inference output     : torch.Size([2, 81, 9])
Per-step count       : 14
Per-step step shape  : torch.Size([2, 81, 9])
Forward pass OK


In [13]:
# Set LOAD_CHECKPOINT to a path string to resume from a saved checkpoint.
# Leave as None to train from scratch.
LOAD_CHECKPOINT = None
# LOAD_CHECKPOINT = 'experiments/checkpoints/trm_sudoku_r001/best_model.pt'

if LOAD_CHECKPOINT and os.path.exists(LOAD_CHECKPOINT):
    ckpt = torch.load(LOAD_CHECKPOINT, map_location=CONFIG['device'])
    model.load_state_dict(ckpt['model_state_dict'])
    _step     = ckpt.get('step', '?')
    _best_acc = ckpt.get('best_val_acc', float('nan'))
    print(f'Loaded checkpoint : {LOAD_CHECKPOINT}')
    print(f'  Step            : {_step}')
    print(f'  Best val acc    : {_best_acc:.4f}')
else:
    print('No checkpoint loaded — training from scratch')

No checkpoint loaded — training from scratch


## Section 5: Training

Uses `TRMTrainer` from `train.py`. The loop calls `trainer.train_step()` per batch and `trainer.evaluate()` per epoch so that per-epoch metrics can be accumulated for later visualization.

In [14]:
trainer = TRMTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
    warmup_steps=CONFIG['warmup_steps'],
    max_steps=CONFIG['max_steps'],
    use_ema=CONFIG['use_ema'],
    ema_decay=CONFIG['ema_decay'],
    device=CONFIG['device'],
    save_dir=CONFIG['checkpoint_dir'],
    task=CONFIG['task'],
)
print('Trainer ready')


Moving model to MPS...
Model on MPS — 530,048 params

Trainer ready


In [ ]:
# Results accumulator — populated each epoch
results = {
    'train_loss':      [],
    'train_acc':       [],
    'train_num_steps': [],
    'train_avg_halt':  [],
    'val_acc':         [],
    'lr':              [],
    'epoch_time':      [],
    'config':          CONFIG,
    'git_hash':        CONFIG.get('git_hash', 'unknown'),
}

best_val_acc    = 0.0
patience_counter = 0
start_time      = time.time()

for epoch in range(CONFIG['num_epochs']):
    epoch_start   = time.time()
    epoch_metrics = defaultdict(list)

    model.train()
    pbar = tqdm(train_loader,
                desc=f'Epoch {epoch+1}/{CONFIG["num_epochs"]}',
                leave=False)
    for batch in pbar:
        m = trainer.train_step(batch)
        for k, v in m.items():
            epoch_metrics[k].append(v)
        pbar.set_postfix(
            loss=f'{m["loss"]:.4f}',
            acc=f'{m["accuracy"]:.3f}',
            lr=f'{trainer.get_lr():.2e}',
        )
        if trainer.step >= CONFIG['max_steps']:
            break

    avg      = {k: sum(v) / max(len(v), 1) for k, v in epoch_metrics.items()}
    val_acc  = trainer.evaluate(val_loader)
    ep_time  = time.time() - epoch_start

    results['train_loss'].append(avg['loss'])
    results['train_acc'].append(avg['accuracy'])
    results['train_num_steps'].append(avg['num_steps'])
    results['train_avg_halt'].append(avg['avg_halt'])
    results['val_acc'].append(val_acc)
    results['lr'].append(trainer.get_lr())
    results['epoch_time'].append(ep_time)

    improved = val_acc > best_val_acc
    marker   = '  ★' if improved else ''
    print(f'Epoch {epoch+1:3d} | '
          f'loss {avg["loss"]:.4f} | '
          f'train {avg["accuracy"]:.3f} | '
          f'val {val_acc:.3f} | '
          f'steps {avg["num_steps"]:.1f} | '
          f'{ep_time:.1f}s{marker}')

    if improved:
        best_val_acc    = val_acc
        patience_counter = 0
        trainer.save_checkpoint('best_model.pt')
    else:
        patience_counter += 1

    if (epoch + 1) % CONFIG['save_every_n_epochs'] == 0:
        trainer.save_checkpoint(f'checkpoint_epoch_{epoch+1}.pt')

    if CONFIG['early_stop_patience'] > 0 and patience_counter >= CONFIG['early_stop_patience']:
        print(f'\nEarly stopping: no improvement for {patience_counter} epochs')
        break

    if trainer.step >= CONFIG['max_steps']:
        print(f'\nReached max_steps ({CONFIG["max_steps"]})')
        break

total_time = time.time() - start_time
results['total_training_time'] = total_time
results['best_val_acc']        = best_val_acc

print(f'\nDone — best val acc: {best_val_acc:.4f}  '
      f'total time: {total_time/60:.1f} min')

Epoch 1/100:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch   1 | loss 2.0962 | train 0.000 | val 0.000 | steps 14.0 | 1852.4s


Epoch 2/100:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch   2 | loss 1.3439 | train 0.000 | val 0.000 | steps 14.0 | 10325.1s


Epoch 3/100:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch   3 | loss 1.1729 | train 0.000 | val 0.000 | steps 14.0 | 1851.8s


Epoch 4/100:   0%|          | 0/782 [00:00<?, ?it/s]

In [ ]:
results_path = f'experiments/results/{CONFIG["experiment_name"]}.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'Results saved: {results_path}')

## Section 6: Evaluation

In [ ]:
best_ckpt_path = os.path.join(CONFIG['checkpoint_dir'], 'best_model.pt')
if os.path.exists(best_ckpt_path):
    trainer.load_checkpoint(best_ckpt_path)
    print(f'Best checkpoint loaded: {best_ckpt_path}')
else:
    print(f'No checkpoint at {best_ckpt_path} — using current model state')

In [ ]:
_eval_kwargs = {}
if CONFIG['task'] == 'maze':
    _eval_kwargs['grid_size'] = CONFIG['grid_size']

metrics = evaluate(model, val_loader, task=CONFIG['task'],
                   device=CONFIG['device'], **_eval_kwargs)

print('=' * 45)
print('EVALUATION RESULTS')
print('=' * 45)
for k, v in metrics.items():
    print(f'  {k:<35}: {v:.4f}')
print('=' * 45)

results['final_metrics'] = metrics

In [ ]:
step_accs = evaluate_per_step(model, val_loader,
                               device=CONFIG['device'], max_batches=20)
print('Per-step accuracy:')
for step, acc in step_accs.items():
    bar = '█' * int(acc * 30)
    print(f'  Step {step:2d}: {acc:.4f}  {bar}')

results['step_accs'] = step_accsd

In [ ]:
model.eval()
_x, _y = val_ds[0]
x_in = _x.unsqueeze(0).to(CONFIG['device'])

with torch.no_grad():
    y_hat, all_preds = model(x_in, training=False, return_all_steps=True)

print(f'Input: {x_in.shape}  |  Supervision steps: {len(all_preds)}\n')

if CONFIG['task'] == 'sudoku':
    puzzle_np = _x.numpy().reshape(9, 9)
    sol_np    = _y.numpy().reshape(9, 9)
    mask      = puzzle_np == 0
    print('Ground truth:')
    print(print_sudoku(puzzle_np, sol_np, sol_np))
    print()
    for i, pred in enumerate(all_preds):
        p   = (pred.argmax(dim=-1)[0] + 1).cpu().numpy().reshape(9, 9)
        acc = (p[mask] == sol_np[mask]).mean()
        print(f'  Step {i:2d}: cell accuracy = {acc:.4f}')
    print('\nFinal prediction:')
    final_p = (y_hat.argmax(dim=-1)[0] + 1).cpu().numpy().reshape(9, 9)
    print(print_sudoku(puzzle_np, final_p, sol_np))
else:
    gs      = CONFIG['grid_size']
    maze_np = _x.numpy().reshape(gs, gs)
    path_np = _y.numpy().reshape(gs, gs)
    print('Ground truth:')
    print(print_maze(maze_np, true_path=path_np))
    print()
    for i, pred in enumerate(all_preds):
        p  = (pred.squeeze(-1) > 0).long()[0].cpu().numpy().reshape(gs, gs)
        tp = int(((p == 1) & (path_np == 1)).sum())
        pr = tp / max(int((p == 1).sum()), 1)
        re = tp / max(int((path_np == 1).sum()), 1)
        print(f'  Step {i:2d}: precision={pr:.3f}  recall={re:.3f}')
    print('\nFinal prediction:')
    final_p = (y_hat.squeeze(-1) > 0).long()[0].cpu().numpy().reshape(gs, gs)
    print(print_maze(maze_np, pred_path=final_p, true_path=path_np))

## Section 7: Visualization & Analysis

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, len(results['train_loss']) + 1)

axes[0].plot(epochs, results['train_loss'], color='steelblue', label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

axes[1].plot(epochs, results['train_acc'], color='steelblue', label='Train Acc')
axes[1].plot(epochs, results['val_acc'],   color='coral',     label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy')
axes[1].legend()

axes[2].plot(epochs, results['train_num_steps'], color='mediumseagreen', label='Avg Steps')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Steps')
axes[2].set_title('Avg Recursion Steps Used')
axes[2].legend()

plt.suptitle(CONFIG['experiment_name'], fontsize=11, y=1.02)
plt.tight_layout()
_save = f'experiments/results/{CONFIG["experiment_name"]}_training_curves.png'
plt.savefig(_save, dpi=150, bbox_inches='tight')
print(f'Saved: {_save}')
plt.show()

In [ ]:
if step_accs:
    _save = f'experiments/results/{CONFIG["experiment_name"]}_per_step.png'
    plot_per_step_accuracy(
        step_accs,
        title=f'{CONFIG["task"].capitalize()} — Accuracy vs Supervision Step',
        save_path=_save,
    )
    print(f'Saved: {_save}')
    plt.show()
else:
    print('No step_accs data — run Section 6 first')

In [ ]:
# Halt probability at each supervision step for a few samples
model.eval()
_x_b, _y_b = next(iter(val_loader))
_x_b = _x_b[:4].to(CONFIG['device'])
_y_b = _y_b[:4].to(CONFIG['device'])

with torch.no_grad():
    _losses, _preds, _halts = model(_x_b, _y_b, training=True)

# _halts: list of tensors, one per supervision step
_halt_vals = []
for h in _halts:
    h = h.detach().cpu()
    if h.dim() == 0:
        _halt_vals.append([h.item()] * _x_b.shape[0])
    else:
        _halt_vals.append(h.view(-1)[:_x_b.shape[0]].tolist())

n_samples = min(4, _x_b.shape[0])
fig, axes = plt.subplots(1, n_samples, figsize=(14, 4), sharey=True)
if n_samples == 1:
    axes = [axes]

for b, ax in enumerate(axes):
    h_curve = [_halt_vals[s][b] for s in range(len(_halt_vals))]
    ax.plot(h_curve, marker='o', color='darkorange', markersize=4)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(f'Sample {b}')
    ax.set_xlabel('Supervision Step')
    ax.set_ylim(0, 1)
    if b == 0:
        ax.set_ylabel('Halt Probability')

plt.suptitle(f'{CONFIG["task"].capitalize()} — Halt Probability per Step')
plt.tight_layout()
_save = f'experiments/results/{CONFIG["experiment_name"]}_confidence_curves.png'
plt.savefig(_save, dpi=150, bbox_inches='tight')
print(f'Saved: {_save}')
plt.show()

In [ ]:
# Scratch pad (y) mean activation heatmap across supervision steps
model.eval()

with torch.no_grad():
    _x_sp = val_ds[0][0].unsqueeze(0).to(CONFIG['device'])
    _x_enc = model.task_head.encode(_x_sp)        # [1, S, D]
    B, S, D = _x_enc.shape

    y_sp = model.y_init.expand(B, S, -1).clone()
    z_sp = model.z_init.expand(B, S, -1).clone()

    y_states = []
    for _ in range(CONFIG['n_supervision']):
        (y_sp, z_sp), _yh, _qh = model.deep_recursion(_x_enc, y_sp, z_sp, with_gradients=False)
        y_states.append(y_sp[0].cpu().numpy())    # [S, D]

# Shape: [n_supervision, S]
y_mean = np.stack([ys.mean(axis=-1) for ys in y_states])

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(y_mean.T, aspect='auto', cmap='RdBu_r', interpolation='nearest')
ax.set_xlabel('Supervision Step')
ax.set_ylabel('Sequence Position (cell index)')
ax.set_title(f'{CONFIG["task"].capitalize()} — Scratch Pad Mean Activation (y)')
plt.colorbar(im, ax=ax, label='Mean activation')
plt.tight_layout()
_save = f'experiments/results/{CONFIG["experiment_name"]}_scratchpad.png'
plt.savefig(_save, dpi=150, bbox_inches='tight')
print(f'Saved: {_save}')
plt.show()

In [ ]:
# Collect wrong predictions from val set
model.eval()
errors = []

with torch.no_grad():
    for _xb, _yb in val_loader:
        _xb = _xb.to(CONFIG['device'])
        _yb = _yb.to(CONFIG['device'])
        _yhat, _all = model(_xb, training=False, return_all_steps=True)
        correct = model.task_head.check_correct(_yhat, _yb, _xb).bool()
        wrong   = (~correct).nonzero(as_tuple=True)[0]
        for idx in wrong[:3]:
            errors.append({
                'x':      _xb[idx].cpu(),
                'y_true': _yb[idx].cpu(),
                'y_pred': _yhat[idx].cpu(),
                'n_steps':len(_all),
            })
        if len(errors) >= 5:
            break

print(f'Error examples collected: {len(errors)}')

if CONFIG['task'] == 'sudoku':
    for i, err in enumerate(errors[:3]):
        puz = err['x'].numpy().reshape(9, 9)
        sol = err['y_true'].numpy().reshape(9, 9)
        pred= (err['y_pred'].argmax(dim=-1) + 1).numpy().reshape(9, 9)
        mask= puz == 0
        acc = (pred[mask] == sol[mask]).mean()
        print(f'\nError {i+1} — cell acc: {acc:.3f}  sup_steps: {err["n_steps"]}')
        print(print_sudoku(puz, pred, sol))
else:
    gs = CONFIG['grid_size']
    for i, err in enumerate(errors[:3]):
        maze= err['x'].numpy().reshape(gs, gs)
        path= err['y_true'].numpy().reshape(gs, gs)
        pred= (err['y_pred'].squeeze(-1) > 0).long().numpy().reshape(gs, gs)
        print(f'\nError {i+1} — sup_steps: {err["n_steps"]}')
        print(print_maze(maze, pred_path=pred, true_path=path))

## Section 8: Ablation Utilities

Template for hyperparameter sweeps. Customize `ABLATION_PARAM` and `ABLATION_VALUES`, then run.

In [ ]:
def run_ablation(
    base_config: dict,
    param_name: str,
    param_values: list,
    ablation_epochs: int = 5,
    ablation_train:  int = 2000,
    ablation_val:    int = 500,
) -> dict:
    """
    Sweep param_name over param_values using short training runs.
    Returns dict: {value -> {val_acc, best_val_acc, final_train_loss, val_accs, config}}

    Customize ablation_epochs / ablation_train / ablation_val as needed.
    These are meant for quick comparisons, not full training runs.
    """
    ablation_results = {}

    for val in param_values:
        cfg = base_config.copy()
        cfg[param_name] = val
        cfg['num_epochs'] = ablation_epochs
        cfg['num_train']  = ablation_train
        cfg['num_val']    = ablation_val
        cfg['max_steps']  = ablation_epochs * (ablation_train // cfg['batch_size']) + 1
        print(f'\n--- {param_name}={val} ---')

        set_seed(cfg['seed'])

        # Dataset
        if cfg['task'] == 'sudoku':
            _dk = {'min_givens': cfg['min_givens'], 'max_givens': cfg['max_givens'],
                   'augment': cfg['augment']}
        else:
            _dk = {'grid_size': cfg['grid_size'], 'augment': cfg['augment']}

        _tr = get_dataset(cfg['task'], 'train', seed=cfg['seed'],
                          num_samples=ablation_train, **_dk)
        _vl = get_dataset(cfg['task'], 'val',   seed=cfg['seed'],
                          num_samples=ablation_val,   **_dk)
        _tl = DataLoader(_tr, batch_size=cfg['batch_size'], shuffle=True,  num_workers=0)
        _vll= DataLoader(_vl, batch_size=cfg['batch_size'], shuffle=False, num_workers=0)

        # Model
        _mk = {k: cfg[k] for k in
               ('task','dim','n_layers','n_heads','n_recursions','n_cycles','n_supervision')}
        if cfg['task'] == 'maze':
            _mk['grid_size'] = cfg['grid_size']
        _m = create_trm_model(**_mk)

        # Trainer
        _t = TRMTrainer(
            model=_m, train_loader=_tl, val_loader=_vll,
            lr=cfg['lr'], weight_decay=cfg['weight_decay'],
            warmup_steps=min(cfg['warmup_steps'], 200),
            max_steps=cfg['max_steps'],
            use_ema=cfg['use_ema'], device=cfg['device'],
            save_dir=f'/tmp/ablation_{param_name}_{val}',
            task=cfg['task'],
        )

        epoch_losses, val_accs = [], []
        for epoch in range(ablation_epochs):
            _m.train()
            _ep_loss = []
            for batch in _tl:
                m = _t.train_step(batch)
                _ep_loss.append(m['loss'])
                if _t.step >= cfg['max_steps']:
                    break
            _va = _t.evaluate(_vll)
            epoch_losses.append(sum(_ep_loss) / max(len(_ep_loss), 1))
            val_accs.append(_va)
            print(f'  ep {epoch+1}: loss={epoch_losses[-1]:.4f}  val={_va:.4f}')
            if _t.step >= cfg['max_steps']:
                break

        ablation_results[val] = {
            'val_acc':          val_accs[-1],
            'best_val_acc':     max(val_accs),
            'final_train_loss': epoch_losses[-1],
            'val_accs':         val_accs,
            'config':           {param_name: val},
        }
        del _m, _t, _tr, _vl

    return ablation_results

print('run_ablation() defined')

In [ ]:
# ── Customize these two lines to run a different sweep ───────────────────────
ABLATION_PARAM  = 'n_recursions'
ABLATION_VALUES = [4, 8, 12]   # three quick runs
# ─────────────────────────────────────────────────────────────────────────────

print(f'Ablation: {ABLATION_PARAM} in {ABLATION_VALUES}')
print('(5 epochs, 2 000 train samples each)\n')

ablation_results = run_ablation(
    base_config=CONFIG,
    param_name=ABLATION_PARAM,
    param_values=ABLATION_VALUES,
    ablation_epochs=5,
    ablation_train=2000,
    ablation_val=500,
)

print(f'\n{"Value":>10} | {"Val Acc":>10} | {"Best Val":>10} | {"Train Loss":>10}')
print('-' * 50)
for v, r in ablation_results.items():
    print(f'{v!r:>10} | {r["val_acc"]:>10.4f} | {r["best_val_acc"]:>10.4f} | {r["final_train_loss"]:>10.4f}')

In [ ]:
def plot_ablation(ablation_results: dict, param_name: str, save_path: str = None):
    """Bar chart of best val acc + val-accuracy learning curves per sweep value."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    values    = list(ablation_results.keys())
    best_accs = [ablation_results[v]['best_val_acc'] for v in values]

    axes[0].bar([str(v) for v in values], best_accs,
                color='steelblue', edgecolor='white')
    axes[0].set_xlabel(param_name)
    axes[0].set_ylabel('Best Val Accuracy')
    axes[0].set_title(f'Ablation: {param_name}')
    for i, acc in enumerate(best_accs):
        axes[0].text(i, acc + 0.001, f'{acc:.4f}',
                     ha='center', va='bottom', fontsize=9)

    for v in values:
        curve = ablation_results[v]['val_accs']
        axes[1].plot(range(1, len(curve) + 1), curve,
                     marker='o', label=f'{param_name}={v}', markersize=4)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Val Accuracy')
    axes[1].set_title('Val Accuracy per Epoch')
    axes[1].legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()

plot_ablation(
    ablation_results,
    param_name=ABLATION_PARAM,
    save_path=f'experiments/results/{CONFIG["experiment_name"]}_ablation_{ABLATION_PARAM}.png',
)

## Section 9: Experiment Logging

Assembles a permanent record of this run: git hash, config, metrics, timing. Appends a one-line entry to `experiments/EXPERIMENT_LOG.md`.

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
hostname  = socket.gethostname()

experiment_record = {
    'experiment_name':            CONFIG['experiment_name'],
    'timestamp':                  timestamp,
    'hostname':                   hostname,
    'git_hash':                   CONFIG.get('git_hash', 'unknown'),
    'config':                     {k: v for k, v in CONFIG.items() if k != 'git_hash'},
    'final_metrics':              results.get('final_metrics', {}),
    'best_val_acc':               results.get('best_val_acc', 0.0),
    'epochs_completed':           len(results.get('train_loss', [])),
    'total_training_time_minutes':results.get('total_training_time', 0.0) / 60,
}

_log_path = f'experiments/logs/{CONFIG["experiment_name"]}_{timestamp}'
try:
    import yaml
    _log_path += '.yaml'
    with open(_log_path, 'w') as f:
        yaml.dump(experiment_record, f, default_flow_style=False, sort_keys=False)
except ImportError:
    _log_path += '.json'
    with open(_log_path, 'w') as f:
        json.dump(experiment_record, f, indent=2, default=str)

print(f'Experiment log saved : {_log_path}')
print(f'  experiment         : {experiment_record["experiment_name"]}')
print(f'  timestamp          : {experiment_record["timestamp"]}')
print(f'  git_hash           : {experiment_record["git_hash"][:8]}')
print(f'  best_val_acc       : {experiment_record["best_val_acc"]:.4f}')
print(f'  training time      : {experiment_record["total_training_time_minutes"]:.1f} min')

In [ ]:
LOG_MD = 'experiments/EXPERIMENT_LOG.md'
_HEADER = (
    '# Experiment Log\n\n'
    '| Tag | Date | Task | Key Change | Val Acc | Notes |\n'
    '|-----|------|------|------------|---------|-------|\n'
)

_metrics_str = '  '.join(
    f'{k}={v:.4f}' for k, v in results.get('final_metrics', {}).items()
)
_line = (
    f'| {CONFIG["experiment_name"]} '
    f'| {datetime.now().strftime("%Y-%m-%d")} '
    f'| {CONFIG["task"]} '
    f'| {CONFIG.get("notes", "baseline run")} '
    f'| {results.get("best_val_acc", 0.0):.4f} '
    f'| {_metrics_str} |\n'
)

if not os.path.exists(LOG_MD):
    with open(LOG_MD, 'w') as f:
        f.write(_HEADER)
        f.write(_line)
    print(f'Created: {LOG_MD}')
else:
    with open(LOG_MD, 'a') as f:
        f.write(_line)
    print(f'Appended to: {LOG_MD}')

print(f'\nLog entry:\n  {_line.strip()}')